# Capstone: The Retrieval Agent

## Module 7 — Capstone Project (Notebook 2 of 6)

The Retriever is Castalia Scholar's **knowledge engine**. Given a research sub-question, it finds relevant documents, scores their relevance, and tracks citations — enabling the downstream Writer to produce properly attributed reports.

### What You'll Build

1. **Document corpus** — 40+ inline research documents spanning AI, computing, and science
2. **FAISS retrieval index** — embed all documents with BGE, store in a vector index
3. **RetrievalAgent** — agentic RAG: initial search → evaluate → reformulate → re-search
4. **Citation tracking** — every result carries full source attribution
5. **Quality assessment** — LLM-judged relevance scoring (1–5 scale)
6. **End-to-end tests** — 5 research sub-questions with full retrieval traces

This notebook implements the `RetrieverInterface` defined in notebook 32 using patterns from the RAG module (Agentic RAG, Self-RAG, CRAG).

---

> **Prerequisites**: Notebook 32 (architecture). Familiarity with rag/ module.
> **Runtime**: Google Colab with T4 GPU.
> **Time**: ~45–60 minutes.

## Part 0: Environment Setup

We need the LLM for query reformulation and relevance judging, plus the embedding model and FAISS for vector search.

In [ ]:
# --- Google Colab Setup ---
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch sentence-transformers faiss-cpu numpy

import torch
import time
import json
import re
import math
import numpy as np
import faiss
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any, Callable, Tuple, Union
from collections import defaultdict
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer

# Mount Google Drive for model caching
drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"
MODEL_NAME = "Qwen/Qwen3-14B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map="auto", quantization_config=quantization_config,
    cache_dir=CACHE_DIR, torch_dtype="auto",
)

# Load embedding model for memory/retrieval
embed_model = SentenceTransformer("BAAI/bge-base-en-v1.5", cache_folder=CACHE_DIR)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs, max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample, top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

def embed_texts(texts):
    """Embed a list of texts using BGE model. Returns numpy array."""
    return embed_model.encode(texts, normalize_embeddings=True)

print(f"✓ LLM loaded: {MODEL_NAME}")
print(f"  GPU memory: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
print(f"✓ Embedding model loaded: BAAI/bge-base-en-v1.5 (768-dim)")

## 1. The Retriever's Job

The Retrieval Agent is more than a simple vector search. It follows the **Agentic RAG** pattern:

```
Query ──► Embed & Search ──► Evaluate Results ──► Good enough?
                                                    │
                                              YES ──┤──► NO
                                               │         │
                                               ▼         ▼
                                          Return     Reformulate query
                                          Results    & search again
                                                         │
                                                         └──► (loop, max 2 retries)
```

### Key Responsibilities
1. **Semantic search** — find documents by meaning, not just keywords
2. **Relevance assessment** — use LLM to judge if results actually answer the question
3. **Query reformulation** — if initial results are poor, rephrase and try again
4. **Citation tracking** — every returned doc carries source metadata for the Writer

### Design Decisions
- **BGE embeddings** — BAAI/bge-base-en-v1.5 (768-dim), good balance of quality and speed
- **FAISS IndexFlatIP** — inner product search on normalized embeddings = cosine similarity
- **Max 2 reformulation cycles** — prevents infinite search loops
- **Relevance threshold** — LLM judges relevance 1–5; docs scoring ≥ 3 are kept

## 2. Building the Document Corpus

We build a corpus of 40+ documents spanning AI, computing, and science topics. Each document has a title, multi-paragraph content, and source attribution — mimicking a real research database.

In [ ]:
# ─── Document Corpus ───
# 40+ research-style documents across AI, computing, and science

CORPUS = [
    {
        "id": "doc_001",
        "title": "Transformer Architecture and Self-Attention",
        "content": "The Transformer architecture, introduced by Vaswani et al. in 2017, replaced recurrent neural networks with a self-attention mechanism that processes all positions in a sequence simultaneously. This parallelization dramatically improved training efficiency on modern hardware.\n\nThe key innovation is multi-head attention, which allows the model to attend to information from different representation subspaces at different positions. Combined with positional encoding, this enables the model to capture both local and long-range dependencies without the vanishing gradient problems of RNNs.",
        "source": "Vaswani et al., Attention Is All You Need, NeurIPS 2017"
    },
    {
        "id": "doc_002",
        "title": "BERT and Bidirectional Pre-training",
        "content": "BERT (Bidirectional Encoder Representations from Transformers) introduced masked language modeling as a pre-training objective. By randomly masking tokens and training the model to predict them from both left and right context, BERT learns deep bidirectional representations.\n\nBERT's pre-train then fine-tune paradigm became the standard approach in NLP. Models pre-trained on large corpora could be fine-tuned on downstream tasks with minimal labeled data, achieving state-of-the-art results on 11 NLP benchmarks simultaneously.",
        "source": "Devlin et al., BERT: Pre-training of Deep Bidirectional Transformers, NAACL 2019"
    },
    {
        "id": "doc_003",
        "title": "GPT and Autoregressive Language Modeling",
        "content": "The GPT series demonstrated that autoregressive language models, trained to predict the next token, scale remarkably well. GPT-2 showed that large language models could generate coherent long-form text, while GPT-3 demonstrated that scaling to 175 billion parameters enables few-shot learning without fine-tuning.\n\nThe in-context learning ability of large autoregressive models was unexpected — by simply providing examples in the prompt, the model could perform tasks it was never explicitly trained on. This emergent capability drove the development of prompt engineering as a discipline.",
        "source": "Brown et al., Language Models are Few-Shot Learners, NeurIPS 2020"
    },
    {
        "id": "doc_004",
        "title": "Model Pruning for Efficient Inference",
        "content": "Neural network pruning removes redundant parameters to create smaller, faster models. Structured pruning removes entire neurons or attention heads, while unstructured pruning zeros out individual weights. The lottery ticket hypothesis suggests that dense networks contain sparse subnetworks that can match full model performance.\n\nModern pruning techniques achieve 80-90% sparsity with minimal accuracy loss. Combined with sparse matrix libraries, pruned models can run significantly faster on commodity hardware, making deployment on edge devices practical.",
        "source": "Frankle & Carlin, The Lottery Ticket Hypothesis, ICLR 2019"
    },
    {
        "id": "doc_005",
        "title": "Quantization: Reducing Model Precision",
        "content": "Quantization reduces the numerical precision of model weights and activations from 32-bit floating point to 8-bit or 4-bit integers. Post-training quantization (PTQ) requires no retraining, while quantization-aware training (QAT) fine-tunes models to be robust to lower precision.\n\nRecent advances in 4-bit quantization (GPTQ, AWQ, bitsandbytes NF4) allow 70B parameter models to run on a single consumer GPU with minimal quality degradation. This democratized access to large language models that previously required multi-GPU clusters.",
        "source": "Dettmers et al., QLoRA: Efficient Finetuning of Quantized LLMs, NeurIPS 2023"
    },
    {
        "id": "doc_006",
        "title": "Knowledge Distillation in Neural Networks",
        "content": "Knowledge distillation trains a smaller 'student' model to mimic the behavior of a larger 'teacher' model. Rather than training on hard labels, the student learns from the teacher's soft probability distributions, which contain richer information about inter-class relationships.\n\nDistillation has been successfully applied to compress BERT-sized models into versions 40-60% smaller with 95%+ task performance retained. The technique is particularly effective when combined with architecture-specific optimizations like attention transfer.",
        "source": "Hinton et al., Distilling the Knowledge in a Neural Network, NIPS Workshop 2015"
    },
    {
        "id": "doc_007",
        "title": "Retrieval-Augmented Generation (RAG)",
        "content": "RAG combines retrieval systems with generative models. Instead of relying solely on parametric knowledge, RAG retrieves relevant documents from an external knowledge base and conditions the generation on them. This reduces hallucination and allows easy knowledge updates.\n\nThe RAG architecture typically uses a dense retriever (like DPR or contriever) to find relevant passages, then feeds them as context to a language model. This separation of knowledge storage from reasoning allows models to access far more information than could fit in parameters alone.",
        "source": "Lewis et al., Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks, NeurIPS 2020"
    },
    {
        "id": "doc_008",
        "title": "Constitutional AI and RLHF",
        "content": "Reinforcement Learning from Human Feedback (RLHF) fine-tunes language models to align with human preferences. A reward model is trained on human comparisons of model outputs, then the language model is optimized against this reward using PPO or DPO.\n\nConstitutional AI extends this by having the model self-critique against a set of principles (a 'constitution'), reducing the need for extensive human annotation. This approach has been shown to produce models that are both helpful and harmless, though reward hacking remains a concern.",
        "source": "Bai et al., Constitutional AI: Harmlessness from AI Feedback, 2022"
    },
    {
        "id": "doc_009",
        "title": "Energy Consumption of Large Language Models",
        "content": "Training large language models requires enormous computational resources. GPT-3's training consumed an estimated 1,287 MWh of energy, equivalent to the annual consumption of 120 US homes. The carbon footprint of training a single large model can exceed 300 tons of CO2.\n\nInference costs are also significant — serving billions of daily queries to models with hundreds of billions of parameters requires massive GPU clusters. The total energy cost of AI inference is estimated to exceed training costs by orders of magnitude when models are widely deployed.",
        "source": "Patterson et al., Carbon Emissions and Large Neural Network Training, 2021"
    },
    {
        "id": "doc_010",
        "title": "Green AI: Efficiency as a Primary Concern",
        "content": "The Green AI movement advocates for measuring and reporting the computational cost of AI research alongside accuracy metrics. The proposal suggests that conference papers include FLOPs, energy consumption, and carbon emissions alongside performance numbers.\n\nGreen AI distinguishes between 'Red AI' (achieving state-of-the-art through scale) and 'Green AI' (achieving good results efficiently). Techniques like efficient architectures, progressive training, and hardware-software co-design are central to the Green AI agenda.",
        "source": "Schwartz et al., Green AI, Communications of the ACM, 2020"
    },
    {
        "id": "doc_011",
        "title": "Mixture of Experts for Efficient Scaling",
        "content": "Mixture of Experts (MoE) models activate only a subset of parameters for each input, allowing models to scale to trillions of parameters while keeping inference cost manageable. A gating network routes each token to the most relevant experts.\n\nGShard and Switch Transformer demonstrated that MoE can achieve better quality-per-FLOP than dense models. Mixtral 8x7B showed that MoE architectures can match or exceed GPT-3.5 performance while using only 13B active parameters per forward pass.",
        "source": "Fedus et al., Switch Transformers: Scaling to Trillion Parameter Models, JMLR 2022"
    },
    {
        "id": "doc_012",
        "title": "Hardware Accelerators for Neural Networks",
        "content": "Custom hardware accelerators like Google's TPU, NVIDIA's Tensor Cores, and specialized AI chips from Cerebras, Graphcore, and SambaNova are designed to maximize throughput for matrix operations central to neural network computation.\n\nTPUv4 pods achieve 1.1 exaflops of peak performance while improving energy efficiency by 2-3x compared to general-purpose GPUs. Neuromorphic chips like Intel's Loihi take a different approach, mimicking brain-like spiking neural networks that can be orders of magnitude more energy-efficient for certain tasks.",
        "source": "Jouppi et al., TPU v4: An Optically Reconfigurable Supercomputer, ISCA 2023"
    },
    {
        "id": "doc_013",
        "title": "Efficient Training Algorithms",
        "content": "Several algorithmic innovations reduce training costs. Mixed-precision training uses FP16 for most operations with FP32 for critical accumulations, halving memory and doubling throughput. Gradient checkpointing trades compute for memory, enabling training of larger models on existing hardware.\n\nMore recently, techniques like LORA (Low-Rank Adaptation) enable fine-tuning large models by training only small adapter matrices (0.1% of parameters), reducing training compute by 10-100x while matching full fine-tuning performance on many tasks.",
        "source": "Hu et al., LoRA: Low-Rank Adaptation of Large Language Models, ICLR 2022"
    },
    {
        "id": "doc_014",
        "title": "Neural Architecture Search (NAS)",
        "content": "Neural Architecture Search automates the design of neural network architectures. Early NAS methods like ENAS and DARTS search over a space of possible architectures using reinforcement learning or gradient-based optimization.\n\nEfficiency-aware NAS includes hardware constraints (latency, energy, memory) in the search objective, producing architectures optimized for specific deployment targets. EfficientNet, designed by NAS, achieved state-of-the-art ImageNet accuracy while being 8x smaller and 6x faster than previous models.",
        "source": "Tan & Le, EfficientNet: Rethinking Model Scaling for CNNs, ICML 2019"
    },
    {
        "id": "doc_015",
        "title": "Sparse Attention Mechanisms",
        "content": "Standard self-attention has O(n²) complexity in sequence length, limiting context windows. Sparse attention patterns (Longformer, BigBird, Reformer) reduce this to O(n log n) or O(n) by attending to only a subset of positions.\n\nFlashAttention takes a different approach: rather than changing the attention pattern, it optimizes the memory access pattern to reduce GPU memory reads/writes by 5-20x, making exact attention practical for much longer sequences. FlashAttention-2 further improved throughput to near theoretical hardware limits.",
        "source": "Dao et al., FlashAttention: Fast and Memory-Efficient Exact Attention, NeurIPS 2022"
    },
    {
        "id": "doc_016",
        "title": "Transfer Learning and Foundation Models",
        "content": "Transfer learning reuses representations learned on one task for another. Foundation models take this to an extreme — a single model pre-trained on vast data becomes the starting point for hundreds of downstream applications.\n\nThe foundation model paradigm is both efficient and concerning. Efficient because pre-training happens once and is amortized across many uses. Concerning because the massive upfront cost creates concentration — only well-resourced organizations can train foundation models, creating dependency for the broader ecosystem.",
        "source": "Bommasani et al., On the Opportunities and Risks of Foundation Models, Stanford HAI, 2021"
    },
    {
        "id": "doc_017",
        "title": "Federated Learning for Privacy-Preserving AI",
        "content": "Federated learning trains models across decentralized data sources without sharing raw data. Each participant trains locally and shares only model updates (gradients), which are aggregated by a central server. This preserves privacy while enabling learning from distributed datasets.\n\nGoogle deployed federated learning at scale for keyboard prediction on Android devices. Recent advances address communication efficiency (compressed gradients), heterogeneous data distributions, and differential privacy guarantees that provably limit information leakage.",
        "source": "McMahan et al., Communication-Efficient Learning of Deep Networks, AISTATS 2017"
    },
    {
        "id": "doc_018",
        "title": "Reinforcement Learning from Human Feedback in Practice",
        "content": "RLHF has become the standard method for aligning language models with human preferences. The process involves three stages: supervised fine-tuning on curated demonstrations, reward model training on human preference rankings, and policy optimization using PPO or similar algorithms.\n\nDirect Preference Optimization (DPO) simplifies this pipeline by eliminating the separate reward model, instead optimizing the language model directly on preference pairs. This reduces training complexity and instability while achieving comparable alignment results.",
        "source": "Rafailov et al., Direct Preference Optimization, NeurIPS 2023"
    },
    {
        "id": "doc_019",
        "title": "Multi-Modal AI Systems",
        "content": "Multi-modal AI systems process and generate content across multiple modalities — text, images, audio, video. Models like GPT-4V, Gemini, and LLaVA demonstrate that vision-language integration enables complex reasoning about visual content.\n\nThe architecture typically involves modality-specific encoders (e.g., CLIP for images, Whisper for audio) connected to a shared language model backbone. This modular approach allows adding new modalities without retraining the entire model, though challenges remain in handling temporal modalities like video efficiently.",
        "source": "OpenAI, GPT-4 Technical Report, 2023"
    },
    {
        "id": "doc_020",
        "title": "Emergent Abilities of Large Language Models",
        "content": "As language models scale, they exhibit abilities not present in smaller models — a phenomenon called emergence. Chain-of-thought reasoning, in-context learning, and instruction following all appear suddenly at certain scale thresholds rather than improving gradually.\n\nThis unpredictability complicates planning: researchers cannot easily predict what capabilities a model will gain from additional scale. Some researchers argue that emergence is partly an artifact of metric choice, while others maintain it represents genuine phase transitions in model capability.",
        "source": "Wei et al., Emergent Abilities of Large Language Models, TMLR 2022"
    },
    {
        "id": "doc_021",
        "title": "Data-Centric AI",
        "content": "Data-centric AI shifts focus from model architecture to data quality. Instead of repeatedly modifying models, practitioners systematically improve the training data — fixing labels, removing noise, adding targeted examples, and balancing distributions.\n\nEmpirical studies show that cleaning 10% of noisy labels improves performance more than doubling model size. Techniques like data augmentation, synthetic data generation, and active learning help create higher-quality datasets with less human effort.",
        "source": "Ng, A., Data-Centric AI Competition, NeurIPS 2021"
    },
    {
        "id": "doc_022",
        "title": "AI Ethics and Bias in Machine Learning",
        "content": "Machine learning systems can amplify societal biases present in training data. Facial recognition systems have shown significantly higher error rates for darker-skinned individuals and women. Language models trained on internet text absorb and reproduce stereotypes, toxic content, and misinformation.\n\nMitigation approaches include debiasing training data, adversarial debiasing during training, post-hoc calibration of outputs, and comprehensive evaluation across demographic groups. However, defining and measuring 'fairness' remains contested — different mathematical definitions of fairness can be mutually exclusive.",
        "source": "Buolamwini & Gebru, Gender Shades: Intersectional Accuracy Disparities, FAT* 2018"
    },
    {
        "id": "doc_023",
        "title": "Autonomous AI Agents and Tool Use",
        "content": "AI agents extend language models with the ability to use tools, maintain state, and act autonomously. Frameworks like ReAct combine reasoning and acting in an interleaved loop: the model thinks about what to do, uses a tool, observes the result, and decides the next action.\n\nRecent systems like AutoGPT, BabyAGI, and Devin demonstrate that LLM-based agents can decompose complex tasks, write and execute code, browse the web, and iteratively work toward goals with minimal human supervision.",
        "source": "Yao et al., ReAct: Synergizing Reasoning and Acting, ICLR 2023"
    },
    {
        "id": "doc_024",
        "title": "Scaling Laws for Neural Language Models",
        "content": "Neural language model performance follows predictable power-law relationships with model size, dataset size, and compute budget. The Chinchilla scaling laws showed that many large models were under-trained: given a fixed compute budget, training a smaller model on more data yields better performance.\n\nThis insight shifted industry practice from the 'bigger is better' paradigm to 'train longer on more data.' Llama models demonstrated that well-trained 7B and 13B models can match or exceed much larger but under-trained models, significantly reducing deployment costs.",
        "source": "Hoffmann et al., Training Compute-Optimal Large Language Models, NeurIPS 2022"
    },
    {
        "id": "doc_025",
        "title": "Prompt Engineering and In-Context Learning",
        "content": "Prompt engineering designs inputs to elicit desired behavior from language models. Techniques range from simple instruction formatting to complex strategies like chain-of-thought prompting, which asks the model to show its reasoning step by step.\n\nFew-shot prompting provides examples of desired input-output pairs within the prompt, enabling task adaptation without parameter updates. This is computationally much cheaper than fine-tuning, though it consumes context window space and results can be sensitive to example selection and ordering.",
        "source": "Wei et al., Chain-of-Thought Prompting Elicits Reasoning, NeurIPS 2022"
    },
    {
        "id": "doc_026",
        "title": "Rule-Based NLP Systems",
        "content": "Before the neural revolution, NLP systems were primarily rule-based. ELIZA (1966) used pattern matching and substitution rules to simulate conversation. SHRDLU (1970) combined a parser with hand-coded world knowledge to understand natural language about a blocks world.\n\nExpert systems in the 1980s encoded domain knowledge as if-then rules. While brittle and expensive to maintain, these systems provided transparent, explainable behavior — a property that modern neural approaches have largely lost. The trade-off between interpretability and capability continues to shape NLP system design.",
        "source": "Jurafsky & Martin, Speech and Language Processing, 3rd Edition"
    },
    {
        "id": "doc_027",
        "title": "Statistical NLP: From Rules to Data",
        "content": "The statistical revolution in NLP replaced hand-crafted rules with probabilistic models learned from data. Hidden Markov Models for part-of-speech tagging, statistical machine translation using phrase tables, and n-gram language models represented a fundamental shift in methodology.\n\nThe key insight was that patterns in large text corpora could capture linguistic regularities that were too complex to encode manually. The famous quote from Frederick Jelinek — 'Every time I fire a linguist, the performance of the speech recognizer goes up' — captured the spirit of this data-driven paradigm shift.",
        "source": "Manning & Schütze, Foundations of Statistical Natural Language Processing, MIT Press"
    },
    {
        "id": "doc_028",
        "title": "Word Embeddings: Word2Vec and GloVe",
        "content": "Word2Vec (2013) showed that training a shallow neural network to predict words from their context (or vice versa) produces dense vector representations that capture semantic relationships. The famous 'king - man + woman = queen' analogy demonstrated that arithmetic on word vectors corresponds to semantic operations.\n\nGloVe (Global Vectors) combined the benefits of count-based and prediction-based methods by factorizing a word co-occurrence matrix. These embeddings became the standard input representation for neural NLP systems until contextual embeddings (ELMo, BERT) superseded them.",
        "source": "Mikolov et al., Efficient Estimation of Word Representations, ICLR Workshop 2013"
    },
    {
        "id": "doc_029",
        "title": "Recurrent Neural Networks for Sequence Modeling",
        "content": "Recurrent Neural Networks (RNNs) process sequences by maintaining a hidden state that is updated at each time step. Long Short-Term Memory (LSTM) networks addressed the vanishing gradient problem with gating mechanisms that control information flow.\n\nLSTMs and GRUs dominated sequence modeling from 2015-2017, achieving state-of-the-art results on machine translation, speech recognition, and text generation. However, their sequential nature prevents parallelization, making them increasingly impractical as datasets and model sizes grew, leading to the Transformer revolution.",
        "source": "Hochreiter & Schmidhuber, Long Short-Term Memory, Neural Computation, 1997"
    },
    {
        "id": "doc_030",
        "title": "Instruction Tuning and Chat Models",
        "content": "Instruction tuning fine-tunes language models on diverse tasks formatted as instructions. FLAN and T0 showed that instruction-tuned models generalize better to unseen tasks than standard pre-trained models. InstructGPT demonstrated that combining instruction tuning with RLHF produces models that follow instructions while being helpful and safe.\n\nThe chat format — alternating user and assistant turns — has become the dominant interface for interacting with language models. This conversational paradigm allows multi-turn reasoning, clarification, and iterative refinement that single-prompt interactions cannot achieve.",
        "source": "Ouyang et al., Training Language Models to Follow Instructions with Human Feedback, NeurIPS 2022"
    },
    {
        "id": "doc_031",
        "title": "Differential Privacy in Machine Learning",
        "content": "Differential privacy provides mathematically rigorous privacy guarantees for machine learning models. By adding calibrated noise during training (DP-SGD), models provably limit the influence of any single training example, preventing extraction of private information.\n\nThe privacy-accuracy trade-off is well-characterized: stronger privacy guarantees require more noise, which degrades model quality. Recent work on large-scale DP training has narrowed this gap, showing that with sufficient data and compute, differentially private models can approach non-private baselines.",
        "source": "Abadi et al., Deep Learning with Differential Privacy, CCS 2016"
    },
    {
        "id": "doc_032",
        "title": "Synthetic Data for Training AI Models",
        "content": "Synthetic data — generated by other AI models or procedural systems — increasingly supplements or replaces human-annotated training data. Self-instruct methods use a strong model to generate training examples for fine-tuning weaker models (e.g., Alpaca, Vicuna).\n\nWhile synthetic data enables rapid dataset creation, risks include mode collapse (the student only learns the teacher's distribution), error amplification (mistakes in generated data become systematic biases), and the 'model collapse' phenomenon where iteratively training on synthetic data degrades quality over generations.",
        "source": "Wang et al., Self-Instruct: Aligning Language Models with Self-Generated Instructions, ACL 2023"
    },
    {
        "id": "doc_033",
        "title": "AI Safety and Alignment Research",
        "content": "AI alignment aims to ensure advanced AI systems behave according to human values and intentions. Key challenges include reward hacking (optimizing proxy metrics instead of true objectives), distributional shift (models behaving unpredictably outside training conditions), and deceptive alignment (models appearing aligned during evaluation but pursuing different goals in deployment).\n\nCurrent approaches include RLHF, constitutional AI, interpretability research (understanding what models have learned), and scalable oversight (using AI to help humans supervise AI). The field is rapidly growing as models become more capable and autonomous.",
        "source": "Ngo et al., The Alignment Problem from a Deep Learning Perspective, 2023"
    },
    {
        "id": "doc_034",
        "title": "Attention Mechanism Variants",
        "content": "Beyond standard dot-product attention, researchers have developed numerous variants. Cross-attention enables models to attend to external context (used in encoder-decoder models and retrieval-augmented systems). Group-query attention (GQA) shares key-value heads across multiple query heads, reducing memory usage.\n\nMulti-query attention (MQA), used in models like PaLM, uses a single key-value head for all query heads, further reducing KV-cache size during inference. These optimizations are critical for deploying large models at scale, where memory bandwidth is often the bottleneck.",
        "source": "Ainslie et al., GQA: Training Generalized Multi-Query Transformer Models, EMNLP 2023"
    },
    {
        "id": "doc_035",
        "title": "AI in Criminal Justice: Risk Assessment",
        "content": "AI-based risk assessment tools like COMPAS are used in criminal justice systems to predict recidivism risk, influencing bail, sentencing, and parole decisions. ProPublica's investigation revealed that COMPAS was significantly more likely to falsely label Black defendants as high-risk compared to white defendants.\n\nThe debate around algorithmic fairness in criminal justice highlights fundamental tensions: predictive accuracy and demographic parity are mathematically incompatible in many settings. Critics argue these tools entrench systemic biases, while proponents claim they are more consistent than individual human judges.",
        "source": "ProPublica, Machine Bias: Risk Assessments in Criminal Sentencing, 2016"
    },
    {
        "id": "doc_036",
        "title": "Explainable AI (XAI) Methods",
        "content": "Explainable AI methods aim to make model decisions interpretable to humans. Post-hoc methods like LIME and SHAP approximate complex models with interpretable local models or compute feature importance scores. Attention visualization shows which input tokens the model attends to.\n\nIntrinsically interpretable models (decision trees, linear models, rule lists) sacrifice some accuracy for transparency. The trade-off between model complexity and interpretability is context-dependent: medical diagnosis may require full transparency, while content recommendation may not.",
        "source": "Ribeiro et al., 'Why Should I Trust You?' Explaining the Predictions of Any Classifier, KDD 2016"
    },
    {
        "id": "doc_037",
        "title": "Speculative Decoding for Faster Inference",
        "content": "Speculative decoding accelerates autoregressive generation by using a small, fast 'draft' model to generate candidate tokens, which a larger 'target' model then verifies in parallel. Since verification is cheaper than generation (it's a single forward pass for multiple tokens), this achieves 2-3x speedup without changing output quality.\n\nThe technique exploits the fact that most tokens in natural language are predictable — the draft model gets them right, and only surprising tokens require the expensive target model. This is particularly effective for tasks like code generation where many tokens are syntactically determined.",
        "source": "Leviathan et al., Fast Inference from Transformers via Speculative Decoding, ICML 2023"
    },
    {
        "id": "doc_038",
        "title": "Continual Learning and Catastrophic Forgetting",
        "content": "Continual learning (lifelong learning) trains models sequentially on new tasks without forgetting previously learned knowledge. Standard neural networks suffer from catastrophic forgetting — training on new data overwrites weights important for old tasks.\n\nApproaches include elastic weight consolidation (slowing updates to important weights), progressive networks (adding capacity for new tasks), and replay methods (rehearsing old examples while learning new ones). Despite progress, continual learning remains significantly harder than training from scratch on all data.",
        "source": "Kirkpatrick et al., Overcoming Catastrophic Forgetting, PNAS 2017"
    },
    {
        "id": "doc_039",
        "title": "Vector Databases and Embedding Search",
        "content": "Vector databases (Pinecone, Weaviate, Milvus, Qdrant) specialize in storing and searching high-dimensional embedding vectors. They support approximate nearest neighbor (ANN) search algorithms like HNSW and IVF that trade small accuracy losses for massive speed improvements over brute-force search.\n\nAs RAG architectures became mainstream, vector databases evolved from specialized tools to core infrastructure. Modern implementations support hybrid search (combining vector similarity with keyword matching), metadata filtering, and real-time index updates, enabling production-grade retrieval systems.",
        "source": "Pan et al., Survey of Vector Database Management Systems, VLDB 2024"
    },
    {
        "id": "doc_040",
        "title": "AI for Scientific Discovery",
        "content": "AI is accelerating scientific discovery across domains. AlphaFold2 predicted protein structures with near-experimental accuracy, solving a 50-year-old grand challenge. GNoME discovered 2.2 million new crystal structures for materials science. AI-driven drug discovery has produced clinical candidates in record time.\n\nThe pattern is consistent: AI excels at searching vast combinatorial spaces that humans cannot explore manually. However, the black-box nature of neural networks means AI-generated hypotheses still require experimental validation, and the risk of learning spurious correlations from scientific datasets remains significant.",
        "source": "Jumper et al., Highly Accurate Protein Structure Prediction with AlphaFold, Nature 2021"
    },
    {
        "id": "doc_041",
        "title": "Edge AI and On-Device Inference",
        "content": "Edge AI runs machine learning models directly on devices like smartphones, IoT sensors, and embedded systems rather than in the cloud. This reduces latency, preserves privacy, and enables offline operation. Mobile-optimized architectures like MobileNet and EfficientNet achieve high accuracy within tight resource constraints.\n\nRecent advances in on-device language models (e.g., Gemma 2B, Phi-3 Mini) bring conversational AI capabilities to mobile devices. Techniques like quantization to 4 bits, structured pruning, and speculative decoding are essential for making these models responsive enough for real-time interaction.",
        "source": "Howard et al., MobileNets: Efficient CNNs for Mobile Vision Applications, CVPR 2017"
    },
    {
        "id": "doc_042",
        "title": "Evaluation Challenges for Language Models",
        "content": "Evaluating language model capabilities is fundamentally difficult. Static benchmarks saturate quickly and suffer from data contamination (test data appearing in training sets). Human evaluation is expensive, subjective, and hard to reproduce. Automatic metrics like BLEU and ROUGE correlate poorly with human judgments for open-ended generation.\n\nModern evaluation approaches include multi-dimensional rubrics (evaluating factuality, helpfulness, safety separately), arena-style comparisons (like Chatbot Arena), and challenge sets designed to probe specific capabilities. The field increasingly recognizes that no single metric captures model quality.",
        "source": "Liang et al., Holistic Evaluation of Language Models (HELM), TMLR 2023"
    },
]

print(f"✓ Document corpus created: {len(CORPUS)} documents")
print(f"  Topics covered:")

# Count documents by rough topic
topic_counts = defaultdict(int)
topic_keywords = {
    "NLP Evolution": ["NLP", "BERT", "GPT", "language model", "word embed", "RNN", "LSTM", "rule-based", "statistical"],
    "Efficiency": ["efficient", "pruning", "quantiz", "distill", "energy", "green", "sparse", "edge", "speculative"],
    "Architecture": ["transformer", "attention", "MoE", "NAS", "multi-modal"],
    "RAG & Search": ["retrieval", "RAG", "vector database", "embedding search"],
    "Safety & Ethics": ["ethics", "bias", "safety", "alignment", "fairness", "privacy", "criminal justice", "explainable"],
    "Training": ["training", "scaling", "instruction tuning", "RLHF", "federated", "continual", "synthetic data"],
    "Applications": ["scientific", "protein", "agent", "tool use"],
}

for doc in CORPUS:
    text = doc["title"] + " " + doc["content"]
    for topic, keywords in topic_keywords.items():
        if any(kw.lower() in text.lower() for kw in keywords):
            topic_counts[topic] += 1
            break

for topic, count in sorted(topic_counts.items(), key=lambda x: -x[1]):
    print(f"    {topic}: {count} documents")

## 3. Building the Retrieval Index

We embed all documents using BGE and store them in a FAISS index for fast similarity search.

In [ ]:
# ─── Build FAISS Index ───

# Prepare texts for embedding (title + content for richer representation)
doc_texts = [f"{doc['title']}. {doc['content']}" for doc in CORPUS]

print(f"Embedding {len(doc_texts)} documents...")
start_time = time.time()
doc_embeddings = embed_texts(doc_texts)
embed_time = time.time() - start_time
print(f"  ✓ Embedded in {embed_time:.1f}s")
print(f"  Embedding shape: {doc_embeddings.shape}")  # (42, 768)

# Build FAISS index (inner product on normalized vectors = cosine similarity)
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(doc_embeddings.astype(np.float32))

print(f"  ✓ FAISS index built: {index.ntotal} vectors, {dimension} dimensions")
print(f"  Index type: IndexFlatIP (exact cosine similarity)")

# Test with a quick query
test_query = "energy efficiency in AI systems"
query_vec = embed_texts([test_query]).astype(np.float32)
scores, indices = index.search(query_vec, 5)
print(f"\nTest query: '{test_query}'")
print(f"  Top 5 results:")
for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), 1):
    print(f"    {rank}. [{score:.3f}] {CORPUS[idx]['title']}")

## 4. Building the RetrievalAgent

The RetrievalAgent implements **Agentic RAG** — it doesn't just search once. It:
1. Embeds the query and searches the index
2. Evaluates result relevance with the LLM
3. If results are insufficient, reformulates the query and searches again
4. Tracks citations for every returned document

In [ ]:
# ─── Citation Dataclass ───

@dataclass
class Citation:
    """Tracks source attribution for a retrieved document."""
    citation_id: int
    doc_id: str
    title: str
    source: str
    excerpt: str = ""

    def format_inline(self) -> str:
        return f"[{self.citation_id}]"

    def format_reference(self) -> str:
        return f"[{self.citation_id}] {self.title}. {self.source}"

@dataclass
class RetrievedDoc:
    """A document with its retrieval metadata."""
    doc_id: str
    title: str
    content: str
    source: str
    relevance_score: float = 0.0
    query: str = ""
    citation: Optional[Citation] = None

    def to_dict(self):
        return {
            "doc_id": self.doc_id, "title": self.title,
            "content": self.content[:200] + "...",
            "source": self.source, "relevance_score": self.relevance_score
        }

print("✓ Citation and RetrievedDoc dataclasses defined")

In [ ]:
# ─── RetrievalAgent: Agentic RAG with Citation Tracking ───

class RetrievalAgent:
    """
    Agentic RAG retrieval agent with iterative search and citation tracking.
    Patterns: Agentic RAG (initial search -> evaluate -> reformulate -> re-search)
    """

    def __init__(self, corpus, index, embed_fn, generate_fn,
                 relevance_threshold=3, max_reformulations=2):
        self.corpus = corpus
        self.index = index
        self.embed_fn = embed_fn
        self.generate_fn = generate_fn
        self.relevance_threshold = relevance_threshold
        self.max_reformulations = max_reformulations
        self.citation_counter = 0
        self.trace = []  # logs each step for debugging

    def _log(self, step, data):
        entry = {"step": step, **data}
        self.trace.append(entry)

    def search(self, query: str, top_k: int = 5) -> List[Dict]:
        """Embed query and search FAISS index."""
        query_vec = self.embed_fn([query]).astype(np.float32)
        scores, indices = self.index.search(query_vec, top_k)
        results = []
        for idx, score in zip(indices[0], scores[0]):
            if idx < len(self.corpus):
                doc = self.corpus[idx].copy()
                doc["similarity_score"] = float(score)
                results.append(doc)
        self._log("search", {"query": query, "results": len(results),
                              "top_score": float(scores[0][0]) if len(scores[0]) > 0 else 0})
        return results

    def assess_relevance(self, doc: Dict, query: str) -> Tuple[int, str]:
        """Use LLM to judge document relevance (1-5 scale)."""
        prompt = f"""Rate the relevance of this document to the research query.

Query: {query}

Document title: {doc['title']}
Document excerpt: {doc['content'][:300]}

Rate relevance from 1-5:
1 = Completely irrelevant
2 = Tangentially related
3 = Somewhat relevant
4 = Highly relevant
5 = Directly answers the query

Respond with ONLY a JSON object: {{"score": <int>, "reason": "<brief reason>"}}"""

        messages = [{"role": "user", "content": prompt}]
        response = self.generate_fn(messages, max_new_tokens=100, temperature=0.1)

        try:
            result = json.loads(response)
            return int(result["score"]), result.get("reason", "")
        except (json.JSONDecodeError, KeyError):
            score_match = re.search(r'"score"s*:s*(d)', response)
            if score_match:
                return int(score_match.group(1)), "parsed from partial response"
            return 3, "could not parse LLM response"

    def reformulate_query(self, original_query: str, results: List[Dict]) -> str:
        """Ask LLM to reformulate query for better results."""
        titles = [r["title"] for r in results[:3]]
        prompt = f"""The following search query didn't return sufficiently relevant results.

Original query: {original_query}
Results found (not relevant enough): {titles}

Reformulate the query to find more relevant documents. Return ONLY the new query string, nothing else."""

        messages = [{"role": "user", "content": prompt}]
        new_query = self.generate_fn(messages, max_new_tokens=60, temperature=0.3).strip()
        new_query = new_query.strip('"').strip("'")
        self._log("reformulate", {"original": original_query, "new": new_query})
        return new_query

    def create_citation(self, doc: Dict) -> Citation:
        """Create a citation for a document."""
        self.citation_counter += 1
        return Citation(
            citation_id=self.citation_counter,
            doc_id=doc["id"],
            title=doc["title"],
            source=doc["source"],
            excerpt=doc["content"][:150]
        )

    def retrieve(self, query: str, top_k: int = 5) -> List[RetrievedDoc]:
        """
        Full agentic retrieval pipeline:
        1. Search index
        2. Assess relevance of results
        3. If insufficient, reformulate and search again
        4. Return results with citations
        """
        self.trace = []
        self._log("start", {"query": query, "top_k": top_k})

        best_results = []
        current_query = query

        for attempt in range(1 + self.max_reformulations):
            attempt_label = "initial" if attempt == 0 else f"reformulation {attempt}"
            self._log("attempt", {"label": attempt_label, "query": current_query})

            # Search
            raw_results = self.search(current_query, top_k=top_k + 2)

            # Assess relevance
            scored_results = []
            for doc in raw_results:
                score, reason = self.assess_relevance(doc, query)  # always judge vs original query
                doc["relevance_score"] = score
                doc["relevance_reason"] = reason
                scored_results.append(doc)
                self._log("assess", {"doc": doc["title"], "score": score, "reason": reason})

            # Keep docs scoring above threshold
            good_results = [d for d in scored_results if d["relevance_score"] >= self.relevance_threshold]

            if len(good_results) >= top_k or attempt == self.max_reformulations:
                best_results = sorted(good_results, key=lambda d: -d["relevance_score"])[:top_k]
                break
            else:
                self._log("insufficient", {"good": len(good_results), "needed": top_k})
                current_query = self.reformulate_query(query, scored_results)

        # Create RetrievedDoc objects with citations
        final_results = []
        for doc in best_results:
            citation = self.create_citation(doc)
            retrieved = RetrievedDoc(
                doc_id=doc["id"], title=doc["title"],
                content=doc["content"], source=doc["source"],
                relevance_score=doc.get("relevance_score", 0),
                query=query, citation=citation
            )
            final_results.append(retrieved)

        self._log("complete", {"returned": len(final_results)})
        return final_results

    def print_trace(self):
        """Print the retrieval trace for debugging."""
        print("\n--- Retrieval Trace ---")
        for entry in self.trace:
            step = entry.pop("step") if "step" in entry else "?"
            print(f"  [{step}] {entry}")


# Create the retrieval agent
retrieval_agent = RetrievalAgent(
    corpus=CORPUS,
    index=index,
    embed_fn=embed_texts,
    generate_fn=generate,
    relevance_threshold=3,
    max_reformulations=2
)

print("✓ RetrievalAgent created")
print(f"  Corpus: {len(CORPUS)} documents")
print(f"  Index: {index.ntotal} vectors")
print(f"  Relevance threshold: 3/5")
print(f"  Max reformulations: 2")

## 5. Testing the Retrieval Agent

We test with 5 research sub-questions that a Planner agent might generate. For each, we examine the retrieval results, relevance scores, and citation tracking.

In [ ]:
# ─── Test 1: Energy Efficiency ───

query1 = "What techniques reduce the energy consumption of training large neural networks?"
print(f"Query: {query1}")
print("="*70)

results1 = retrieval_agent.retrieve(query1, top_k=5)

print(f"\nRetrieved {len(results1)} documents:")
for r in results1:
    print(f"  {r.citation.format_inline()} {r.title}")
    print(f"     Relevance: {r.relevance_score}/5 | Source: {r.source[:50]}...")
    print()

print("Citations:")
for r in results1:
    print(f"  {r.citation.format_reference()}")

retrieval_agent.print_trace()

In [ ]:
# ─── Test 2: Model Compression ───

query2 = "How do model compression techniques like pruning and quantization work?"
print(f"Query: {query2}")
print("="*70)

results2 = retrieval_agent.retrieve(query2, top_k=5)

print(f"\nRetrieved {len(results2)} documents:")
for r in results2:
    print(f"  {r.citation.format_inline()} {r.title}")
    print(f"     Relevance: {r.relevance_score}/5 | Source: {r.source[:50]}...")
    print()

In [ ]:
# ─── Test 3: NLP Evolution ───

query3 = "How has NLP evolved from rule-based systems to neural approaches?"
print(f"Query: {query3}")
print("="*70)

results3 = retrieval_agent.retrieve(query3, top_k=5)

print(f"\nRetrieved {len(results3)} documents:")
for r in results3:
    print(f"  {r.citation.format_inline()} {r.title}")
    print(f"     Relevance: {r.relevance_score}/5")
    print()

In [ ]:
# ─── Test 4: AI Ethics ───

query4 = "What are the ethical concerns with AI bias in automated decision-making?"
print(f"Query: {query4}")
print("="*70)

results4 = retrieval_agent.retrieve(query4, top_k=5)

print(f"\nRetrieved {len(results4)} documents:")
for r in results4:
    print(f"  {r.citation.format_inline()} {r.title}")
    print(f"     Relevance: {r.relevance_score}/5")
    print()

In [ ]:
# ─── Test 5: Efficient Architectures ───

query5 = "What architectural innovations make AI models more computationally efficient?"
print(f"Query: {query5}")
print("="*70)

results5 = retrieval_agent.retrieve(query5, top_k=5)

print(f"\nRetrieved {len(results5)} documents:")
for r in results5:
    print(f"  {r.citation.format_inline()} {r.title}")
    print(f"     Relevance: {r.relevance_score}/5")
    print()

# Show full trace for last query
retrieval_agent.print_trace()

## 6. Quality Assessment — Analyzing Retrieval Performance

Let's evaluate how well the retrieval agent performs across all our test queries.

In [ ]:
# ─── Retrieval Quality Analysis ───

all_tests = [
    ("Energy efficiency in training", results1),
    ("Model compression", results2),
    ("NLP evolution", results3),
    ("AI ethics and bias", results4),
    ("Efficient architectures", results5),
]

print("Retrieval Quality Summary")
print("="*70)
print(f"{'Query':<35} {'Docs':>5} {'Avg Score':>10} {'Min':>5} {'Max':>5}")
print("-"*70)

total_docs = 0
total_score = 0
all_scores = []

for name, results in all_tests:
    if results:
        scores = [r.relevance_score for r in results]
        avg = sum(scores) / len(scores)
        mn, mx = min(scores), max(scores)
        total_docs += len(results)
        total_score += sum(scores)
        all_scores.extend(scores)
        print(f"  {name:<33} {len(results):>5} {avg:>10.1f} {mn:>5.0f} {mx:>5.0f}")
    else:
        print(f"  {name:<33}     0          -     -     -")

print("-"*70)
if all_scores:
    overall_avg = sum(all_scores) / len(all_scores)
    print(f"  {'OVERALL':<33} {total_docs:>5} {overall_avg:>10.1f} {min(all_scores):>5.0f} {max(all_scores):>5.0f}")

print(f"\nTotal citations generated: {retrieval_agent.citation_counter}")
print(f"Score distribution:")
from collections import Counter
score_dist = Counter(int(s) for s in all_scores)
for score in sorted(score_dist.keys()):
    bar = "█" * score_dist[score]
    print(f"  Score {score}: {bar} ({score_dist[score]})")

## 7. Integration Notes

### How the Retriever Fits Into Castalia Scholar

The Retrieval Agent is called by the Orchestrator after the Planner has decomposed the research question:

```
Planner output: ["sub_q1", "sub_q2", "sub_q3"]
    │
    ▼ (for each sub-question)
Retriever.retrieve(sub_q) ──► [RetrievedDoc, RetrievedDoc, ...]
    │                            (each with Citation attached)
    ▼
All docs collected ──► Analyzer
```

### Connection to RAG Module Patterns

| Pattern | How We Use It |
|---------|--------------|
| **Agentic RAG** | The retrieve loop: search → assess → reformulate → search again |
| **Self-RAG** | LLM judges document relevance (self-assessment) |
| **CRAG** | Query reformulation when results are insufficient |

### What the Analyzer Expects

The Analysis Agent (notebook 34) receives the retrieved documents and needs:
- Document content for synthesis
- Source attribution for contradiction detection
- Relevance scores for evidence weighing

## Key Takeaways

1. **Agentic RAG > basic RAG** — iterative search with relevance assessment and query reformulation finds better results than single-shot vector search.
2. **Citation tracking is essential** — every retrieved document carries full source metadata, enabling proper attribution in the final report.
3. **LLM-as-judge for relevance** — vector similarity alone isn't enough; LLM assessment catches semantic relevance that embedding distance misses.
4. **Bounded iteration** — max 2 reformulations prevents infinite search loops while allowing recovery from poor initial queries.
5. **Structured output** — `RetrievedDoc` and `Citation` dataclasses make the interface between Retriever and Analyzer explicit and type-safe.

> **Next notebook**: We build the **Analysis Agent** — synthesis, contradiction detection, and gap identification.